# Embeddings & Cosine Similarity

An embedding maps text to a fixed-length vector such that semantically similar text ends up with similar (high cosine similarity) vectors. This is the mechanism RAG retrieval in this project is built on (`agent-service/app/rag/embeddings.py`, `retriever.py`).

**Network note:** same constraint as the tokenization notebook — the real `sentence-transformers` model download is blocked in this sandbox. The embedding math (cosine similarity, what makes a good vs. bad embedding) is demonstrated for real below using this project's own offline `local-hash` embedding provider — a deterministic, zero-network fallback, *not* a quality replacement for a trained model (see the docstring in `embeddings.py`).

In [1]:
import sys
sys.path.insert(0, '../agent-service')

from app.config import Settings
from app.rag.embeddings import embed_texts

settings = Settings(embedding_provider='local-hash', embedding_dim=128)

sentences = [
    'payment service is failing in production',
    'checkout service is down in prod',
    'the cat sat on the mat',
]
vectors = embed_texts(sentences, settings)
print('embedding dimension:', len(vectors[0]))

embedding dimension: 128


In [2]:
import math

def cosine_similarity(a, b):
    dot = sum(x * y for x, y in zip(a, b))
    norm_a = math.sqrt(sum(x * x for x in a))
    norm_b = math.sqrt(sum(x * x for x in b))
    return dot / (norm_a * norm_b) if norm_a and norm_b else 0.0

for i in range(len(sentences)):
    for j in range(i + 1, len(sentences)):
        sim = cosine_similarity(vectors[i], vectors[j])
        print(f'{sim:.3f}  {sentences[i]!r}  <->  {sentences[j]!r}')

0.500  'payment service is failing in production'  <->  'checkout service is down in prod'
0.000  'payment service is failing in production'  <->  'the cat sat on the mat'
0.000  'checkout service is down in prod'  <->  'the cat sat on the mat'


With a real trained embedding model, sentence 1 and 2 (both about a production service outage, different vocabulary) would score much higher against each other than either scores against the unrelated third sentence — that's the semantic generalization a trained model provides. `local-hash` is a bag-of-hashed-tokens vector: it can only recognize *shared words*, not synonyms or paraphrases, which is exactly why it's documented as a mechanism-testing fallback, not a production embedding provider.

In [3]:
from sentence_transformers import SentenceTransformer

try:
    model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
    real_vectors = model.encode(sentences, normalize_embeddings=True)
    print('embedding dimension:', real_vectors.shape[1])
    for i in range(len(sentences)):
        for j in range(i + 1, len(sentences)):
            sim = float(real_vectors[i] @ real_vectors[j])
            print(f'{sim:.3f}  {sentences[i]!r}  <->  {sentences[j]!r}')
except Exception as exc:
    print('Could not reach huggingface.co from this sandbox (expected here):')
    print(f'  {type(exc).__name__}: {exc}')
    print()
    print('On a normal-network machine this cell shows sentence 1 & 2')
    print('scoring ~0.6-0.8 against each other and ~0.05-0.15 against')
    print('the unrelated third sentence — the semantic gap the toy')
    print('hashing embedding above cannot capture.')

/home/user/ai-agent/agent-service/.venv/lib/python3.11/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:13: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm, trange


'(MaxRetryError("HTTPSConnectionPool(host='huggingface.co', port=443): Max retries exceeded with url: /sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json (Caused by ProxyError('Unable to connect to proxy', OSError('Tunnel connection failed: 403 Forbidden')))"), '(Request ID: 8765c034-6974-40c3-88e1-4cafa2b81a61)')' thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json


Retrying in 1s [Retry 1/5].


'(MaxRetryError("HTTPSConnectionPool(host='huggingface.co', port=443): Max retries exceeded with url: /sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json (Caused by ProxyError('Unable to connect to proxy', OSError('Tunnel connection failed: 403 Forbidden')))"), '(Request ID: 096b0d38-963e-4db3-90e0-719207129234)')' thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json


Retrying in 2s [Retry 2/5].


...

Retrying in 8s [Retry 4/5].


'(MaxRetryError("HTTPSConnectionPool(host='huggingface.co', port=443): Max retries exceeded with url: /sentence-transformers/all-MiniLM-L6-v2/resolve/main/config.json (Caused by ProxyError('Unable to connect to proxy', OSError('Tunnel connection failed: 403 Forbidden')))"), '(Request ID: 49c2161f-c09b-4e68-b327-6a913d444df6)')' thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config.json


Retrying in 8s [Retry 5/5].


'(MaxRetryError("HTTPSConnectionPool(host='huggingface.co', port=443): Max retries exceeded with url: /sentence-transformers/all-MiniLM-L6-v2/resolve/main/config.json (Caused by ProxyError('Unable to connect to proxy', OSError('Tunnel connection failed: 403 Forbidden')))"), '(Request ID: 53eb2162-3ada-49dc-96b6-d77f3e91c13e)')' thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config.json


Could not reach huggingface.co from this sandbox (expected here):
  ProxyError: (MaxRetryError("HTTPSConnectionPool(host='huggingface.co', port=443): Max retries exceeded with url: /sentence-transformers/all-MiniLM-L6-v2/resolve/main/config.json (Caused by ProxyError('Unable to connect to proxy', OSError('Tunnel connection failed: 403 Forbidden')))"), '(Request ID: 53eb2162-3ada-49dc-96b6-d77f3e91c13e)')

On a normal-network machine this cell shows sentence 1 & 2
scoring ~0.6-0.8 against each other and ~0.05-0.15 against
the unrelated third sentence — the semantic gap the toy
hashing embedding above cannot capture.
